# ch03 Final
This notebook contains a clean, usable implementation of self-attention, causal masking, and multi-head attention. It is organized for learning and reuse rather than experimentation.

In [ ]:
import torch
import torch.nn as nn
torch.set_printoptions(sci_mode=False)

## Basic dot-product attention
We compute raw attention scores with query-key dot products, normalize with softmax, and use the resulting weights to combine values.

In [ ]:
inputs = torch.randn(6, 3)  # 6 tokens, embedding size 3
print('inputs shape:', inputs.shape)

query = inputs[1]  # use token 1 as the query
attn_scores = inputs @ query
attn_weights = torch.softmax(attn_scores, dim=0)
context = attn_weights.unsqueeze(-1) * inputs
context = context.sum(dim=0)

print('attention scores:', attn_scores)
print('attention weights:', attn_weights)
print('context vector:', context)

## Trainable self-attention module
This class learns query, key, and value projections and computes attention for all tokens in one pass.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.scale = d_out ** 0.5

    def forward(self, x):
        query = self.W_query(x)
        key = self.W_key(x)
        value = self.W_value(x)
        attn_scores = query @ key.transpose(-2, -1)
        attn_weights = torch.softmax(attn_scores / self.scale, dim=-1)
        context = attn_weights @ value
        return context, attn_weights

sa = SelfAttention(d_in=3, d_out=2)
context, weights = sa(inputs)
print('context shape:', context.shape)
print('attention weights shape:', weights.shape)

## Causal masking
Causal masking prevents each position from attending to future positions. We use an upper-triangle mask and fill masked entries with -inf before softmax.

In [ ]:
def causal_mask(seq_len, device=None):
    mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
    return mask

mask = causal_mask(6, device=inputs.device)
print(mask.int())

# Example causal attention scores
query = inputs
key = inputs
scores = query @ key.transpose(-2, -1)
scores = scores.masked_fill(mask, float('-inf'))
weights = torch.softmax(scores / (inputs.shape[-1] ** 0.5), dim=-1)
print('masked weights[0]:', weights[0])

## Causal self-attention module
This version creates a dynamic causal mask from the input sequence length and applies it inside the forward pass.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.scale = d_out ** 0.5

    def forward(self, x):
        batch, seq_len, _ = x.shape
        query = self.W_query(x)
        key = self.W_key(x)
        value = self.W_value(x)
        attn_scores = query @ key.transpose(-2, -1)
        mask = causal_mask(seq_len, device=x.device)
        attn_scores = attn_scores.masked_fill(mask.unsqueeze(0), float('-inf'))
        attn_weights = torch.softmax(attn_scores / self.scale, dim=-1)
        context = attn_weights @ value
        return context, attn_weights

batch_inputs = torch.stack((inputs, inputs), dim=0)
causal_sa = CausalSelfAttention(d_in=3, d_out=2)
context, weights = causal_sa(batch_inputs)
print('causal context shape:', context.shape)
print('causal weights shape:', weights.shape)

## Multi-Head Attention
Multi-head attention splits the projected embeddings into several heads, computes attention in parallel, and concatenates the results.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = self.head_dim ** 0.5
        self.W_query = nn.Linear(d_in, d_model, bias=False)
        self.W_key = nn.Linear(d_in, d_model, bias=False)
        self.W_value = nn.Linear(d_in, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch, seq_len, _ = x.shape
        query = self.W_query(x)
        key = self.W_key(x)
        value = self.W_value(x)
        query = query.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        key = key.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        value = value.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        attn_scores = query @ key.transpose(-2, -1)
        mask = causal_mask(seq_len, device=x.device)
        attn_scores = attn_scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn_weights = torch.softmax(attn_scores / self.scale, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context = attn_weights @ value
        context = context.transpose(1, 2).contiguous().view(batch, seq_len, -1)
        return self.out_proj(context), attn_weights

mha = MultiHeadAttention(d_in=3, d_model=4, num_heads=2, dropout=0.0)
output, weights = mha(batch_inputs)
print('multi-head output shape:', output.shape)
print('multi-head weights shape:', weights.shape)

## Notes
- `d_model` is the final model dimension after concatenating all heads.
- For causal attention, the mask ensures position `i` cannot attend to positions `j > i`.
- `torch.nn.MultiheadAttention` provides the same behavior in a production-ready module.